In [14]:
from google import genai
from google.genai import types
import pandas as pd
import google.generativeai as genai
import json
import os.path

import gspread
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from datetime import datetime, timedelta

def main():
    today = datetime.today()
    first_day_this_month = today.replace(day=1)
    last_day_prev_month = first_day_this_month - timedelta(days=1)
    
    # 1. Fetch filtered PO and STR rows
    df = get_looker_raw_data()
    df = get_normalized_capex(df)
    print(df)
    
    # 2. Define target spreadsheet details
    spreadsheet_id = '1qzRhKHJ9WEKkII6oijz5g4-fLruU0xcb61kMtK8y_5A'
    target_range = "'Normalized Data'!A1"
    
    # 3. Connect and execute overwrite replacement
    creds = connect_to_sheet()
    replace_sheet_content(spreadsheet_id, target_range, creds, df)
    # Convert month name to a 2026 datetime object, then shift to Month End
    

def get_looker_raw_data():
    creds = connect_to_sheet()
    client = gspread.authorize(creds)
    
    spreadsheet_id = '1qzRhKHJ9WEKkII6oijz5g4-fLruU0xcb61kMtK8y_5A'
    sheet = client.open_by_key(spreadsheet_id)
    worksheet = sheet.worksheet("RAW DATA FOR LOOKER")
    
    # 1. Fetch the entire continuous block from Column G to CX
    full_range = 'G1:CX1186'
    raw_values = worksheet.get(full_range)
    
    if not raw_values:
        print("No data found.")
        return None
        
    # 2. Separate headers from data rows
    headers = [str(h).strip() for h in raw_values[0]]
    data_rows = raw_values[1:]
    
    # Pad any short rows with empty strings to prevent alignment shifts
    max_len = len(headers)
    cleaned_data = [row + [''] * (max_len - len(row)) for row in data_rows]
    
    # 3. Create initial full DataFrame
    df_full = pd.DataFrame(cleaned_data, columns=headers)
    
    # 4. Identify columns we want to keep
    # Dynamically locate the Tipe Capex column header
    tipe_capex_col = next((col for col in df_full.columns if 'TIPE CAPEX' in col), None)
    
    columns_to_keep = ['NAMA CAPEX']
    if tipe_capex_col:
        columns_to_keep.append(tipe_capex_col)
    
    if 'Kategori Right Issue' in df_full.columns:
        columns_to_keep.append('Kategori Right Issue')
        
    # Gather all columns starting with Plan or Real
    monthly_cols = [col for col in df_full.columns if col.startswith('Plan') or col.startswith('Real')]
    columns_to_keep.extend(monthly_cols)
    
    # 5. Filter DataFrame to contain ONLY your target columns
    df_filtered = df_full[columns_to_keep]
    
    return df_filtered

def get_normalized_capex(df):
    df.columns = df.columns.str.strip()
    
    # 2. Dynamically find the TIPE CAPEX column
    tipe_capex_col = None
    for col in df.columns:
        if 'TIPE CAPEX' in col:
            tipe_capex_col = col
            break
            
    if tipe_capex_col is None:
        raise KeyError("Could not find any column containing 'TIPE CAPEX' in the raw sheet data.")
        
    df = df.rename(columns={tipe_capex_col: 'TIPE CAPEX'})
    
    # Include Kategori Right Issue into our structural base anchor columns
    base_cols = ['NAMA CAPEX', 'TIPE CAPEX']
    if 'Kategori Right Issue' in df.columns:
        base_cols.append('Kategori Right Issue')
    
    # 3. Dynamically gather all Plan and Real monthly columns
    plan_cols = [col for col in df.columns if col.startswith('Plan')]
    real_cols = [col for col in df.columns if col.startswith('Real')]
    
    # 4. Melt Plan columns
    df_plan = df.melt(id_vars=base_cols, value_vars=plan_cols, var_name='Month', value_name='Plan')
    df_plan['Month'] = df_plan['Month'].str.replace('Plan ', '', case=False).str.strip()
    
    # 5. Melt Real columns
    df_real = df.melt(id_vars=base_cols, value_vars=real_cols, var_name='Month', value_name='Real')
    df_real['Month'] = df_real['Month'].str.replace('Real ', '', case=False).str.strip()
    
    # 6. Merge both structures side-by-side
    df_normalized = pd.merge(df_plan, df_real, on=base_cols + ['Month'], how='outer')
    
    # Clean string spaces for accurate filtering
    df_normalized['TIPE CAPEX'] = df_normalized['TIPE CAPEX'].astype(str).str.strip()
    df_normalized['NAMA CAPEX'] = df_normalized['NAMA CAPEX'].astype(str).str.strip()
    if 'Kategori Right Issue' in df_normalized.columns:
        df_normalized['Kategori Right Issue'] = df_normalized['Kategori Right Issue'].astype(str).str.strip()
    
    
    # Filter out empty names and invalid historical summary rows
    invalid_months = ['Realisasi Spending 2023', 'Realisasi Spending 2024']
    df_normalized = df_normalized[~df_normalized['Month'].isin(invalid_months)]
    df_normalized = df_normalized[df_normalized['NAMA CAPEX'] != '']
    
    # 8. Clean numeric currency and thousands values
    for col in ['Plan', 'Real']:
        df_normalized[col] = df_normalized[col].astype(str).str.strip()
        df_normalized[col] = df_normalized[col].replace(['-', ''], '0')
        df_normalized[col] = df_normalized[col].str.replace('.', '', regex=False)
        df_normalized[col] = pd.to_numeric(df_normalized[col], errors='coerce').fillna(0).astype(float)
        
    # --- Convert Indonesian Month Name to a Real Date String (YYYY-MM-20) ---
    month_mapping = {
        'Januari': '01', 'Feb': '02', 'Februari': '02', 'March': '03', 'Maret': '03',
        'April': '04', 'Mei': '05', 'May': '05', 'June': '06', 'Juni': '06',
        'July': '07', 'Juli': '07', 'Agustus': '08', 'August': '08',
        'September': '09', 'Oktober': '10', 'October': '10',
        'November': '11', 'Desember': '12', 'December': '12'
    }
    
    df_normalized['Month'] = df_normalized['Month'].str.capitalize().str.strip()
    df_normalized['Month_Num'] = df_normalized['Month'].map(month_mapping)
    df_normalized['Month'] = '2026-' + df_normalized['Month_Num'] + '-20'
    df_normalized = df_normalized.drop(columns=['Month_Num'])
        
    # 9. Reorder layout to your exact 5 columns (Dropping TIPE CAPEX but including Kategori Right Issue)
    final_cols = ['NAMA CAPEX', 'TIPE CAPEX', 'Kategori Right Issue', 'Month', 'Real', 'Plan']
    # Fallback layout check if the column was missing entirely from the source sheet
    if 'Kategori Right Issue' not in df_normalized.columns:
        final_cols.remove('Kategori Right Issue')
        
    df_normalized = df_normalized[final_cols]
    
    return df_normalized

def replace_sheet_content(spreadsheetId, spread_range, creds, df):
    """
    Clears the specified range and completely overwrites it with 
    the provided DataFrame content (including headers).
    """
    # Fill NaN values with empty strings to prevent API errors
    df_clean = df.fillna('')
    header = df_clean.columns.tolist()
    data = df_clean.values.tolist()
    
    # Combine header row and data rows
    values_with_header = [header] + data
    
    try:
        service = build('sheets', 'v4', credentials=creds)
        
        # 1. Clear existing data in the target sheet first to avoid residual old data
        sheet_name = spread_range.split('!')[0].replace("'", "")
        service.spreadsheets().values().clear(
            spreadsheetId=spreadsheetId,
            range=f"'{sheet_name}'!A1:Z"
        ).execute()
        
        # 2. Write the fresh data structure
        body = {'values': values_with_header}
        result = service.spreadsheets().values().update(
            spreadsheetId=spreadsheetId,
            range=spread_range,
            valueInputOption="USER_ENTERED",
            body=body
        ).execute()
        
        print(f"Successfully replaced sheet! {result.get('updatedCells')} cells written.")
        return result
        
    except HttpError as error:
        print(f"An error occurred: {error}")
        return error
    
def connect_to_sheet():
    SCOPES = ['https://www.googleapis.com/auth/spreadsheets']
    creds = None
    # The file token.json stores the user's access and refresh tokens, and is
    # created automatically when the authorization flow completes for the first
    # time.
    if os.path.exists(r'C:\Users\mochamad.ilmawan\token.json'):
        creds = Credentials.from_authorized_user_file(r'C:\Users\mochamad.ilmawan\token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                'D:\OneDrive\Documents\Direct Selling\MONITORING\Monitoring 2023\client_secret_738271294618-6g8e0hle4jpq8nau1d7b3q9jfsgp0ruk.apps.googleusercontent.com.json', SCOPES)
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open(r'C:\Users\mochamad.ilmawan\token.json', 'w') as token:
            token.write(creds.to_json())
    return creds

def get_excel_data(address, sheet_name=0, usecols=None, skiprows=0, nrows=None, header=None):
    df = pd.read_excel(
        address,
        sheet_name=sheet_name,
        usecols=usecols,
        skiprows=skiprows,
        nrows=nrows,
        dtype=str,
        thousands='.', 
        decimal=','
    )
    return df

def append_sheet(spreadSheetId, spread_range, scopes, creds, df):
    df_clean = df.fillna('')
    header = df.columns.tolist()
    data = df.values.tolist()
    
    # Combine them: [header] creates a list within a list to match the Sheets API format
    values_with_header = [header] + data
    try:
        values = values_with_header
        service = build('sheets', 'v4', credentials=creds)
        body = {
            'values': values_with_header
        }
        result = service.spreadsheets().values().append(
            spreadsheetId=spreadSheetId, 
            range=spread_range, 
            valueInputOption="USER_ENTERED", 
            body=body).execute()
        print(f"{(result.get('updates').get('updatedCells'))} cells appended.")
        return result

    except HttpError as error:
        print(f"An error occurred: {error}")
        return error
    
if __name__ == '__main__':
    main()

                               NAMA CAPEX TIPE CAPEX Kategori Right Issue  \
461584  KANTOR PUSAT SEMEN GRESIK TAHAP 1        PKO                        
461585  KANTOR PUSAT SEMEN GRESIK TAHAP 1        PKO                        
461586  KANTOR PUSAT SEMEN GRESIK TAHAP 1        PKO                        
461587  KANTOR PUSAT SEMEN GRESIK TAHAP 1        PKO                        
461588  KANTOR PUSAT SEMEN GRESIK TAHAP 1        PKO                        
...                                   ...        ...                  ...   
475511                     sertifikasi SQ        PKO                        
475512                     sertifikasi SQ        PKO                        
475513                     sertifikasi SQ        PKO                        
475514                     sertifikasi SQ        PKO                        
475517                     sertifikasi SQ        PKO                        

             Month  Real          Plan  
461584  2026-08-20   0.0  0.000000